In [ ]:
# =========================
# 1. Install Libraries
# =========================

!pip install -q torch torchvision torchaudio
!pip install -q pandas matplotlib scikit-learn pillow
!pip install tqdm scikit-learn

In [ ]:
# =========================
# 1. Imports
# =========================

import os
import json
import random
import numpy as np
from PIL import Image
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

In [ ]:
# =========================
# 2. Seed
# =========================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# =========================
# 3. Device
# =========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip "/content/drive/MyDrive/balanced_dataset.zip" -d "/content/"

In [ ]:
import os

print(os.listdir("/content/content/balanced_dataset"))

In [ ]:
# =========================
# 4. Config
# =========================

DATA_DIR    = "/content/content/balanced_dataset"
BATCH_SIZE  = 64
EPOCHS      = 40
PATIENCE    = 8
NUM_WORKERS = 2

# Differential LRs — layer2 gets smallest LR
LR_LAYER2   = 1e-5
LR_LAYER3   = 5e-5
LR_LAYER4   = 1e-4
LR_FC       = 1e-3



In [ ]:
# =========================
# 5. Class Mapping
# =========================

train_json = f"{DATA_DIR}/train_data.json"
labels_set = set()

with open(train_json, "r") as f:
    for line in f:
        item = json.loads(line)
        labels_set.add(item["class_label"])

classes      = sorted(list(labels_set))
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

print(f"Total Classes: {len(classes)}")
print(classes)

In [ ]:
# =========================
# 6. Focal Loss
#    Focuses training on hard/
#    misclassified examples
#    Great for confused classes
# =========================

class FocalLoss(nn.Module):

    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.1):
        super().__init__()
        self.gamma           = gamma
        self.weight          = weight
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        # Standard CE with smoothing
        ce_loss = F.cross_entropy(
            inputs, targets,
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction="none"
        )
        pt      = torch.exp(-ce_loss)
        focal   = ((1 - pt) ** self.gamma) * ce_loss
        return focal.mean()

In [ ]:
# =========================
# 7. Dataset Class
# =========================

class IndoFashionDataset(Dataset):

    def __init__(self, json_path, base_path, transform=None):
        self.data = []
        with open(json_path, "r") as f:
            for line in f:
                self.data.append(json.loads(line))
        self.base_path = base_path
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item       = self.data[idx]
        image_path = os.path.join(self.base_path, item["image_path"])
        image      = Image.open(image_path).convert("RGB")
        label      = class_to_idx[item["class_label"]]
        if self.transform:
            image = self.transform(image)
        return image, label

    def get_labels(self):
        return [class_to_idx[json.loads(line)["class_label"]]
                for line in open(f"{self.base_path}/train_data.json")]

In [ ]:
# =========================
# 8. Transforms
# =========================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(20),
    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.4,
        hue=0.15
    ),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.25)),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# 6 TTA views — more views = better averaging
tta_transforms = [

    # View 1 — standard
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),

    # View 2 — horizontal flip
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),

    # View 3 — center crop
    transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),

    # View 4 — random crop
    transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),

    # View 5 — slight rotation + center crop
    transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomRotation(degrees=(10, 10)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),

    # View 6 — color jitter
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
]

In [ ]:
# =========================
# 9. Compute Class Weights
#    Upweights confused/rare classes
# =========================

all_train_labels = []
with open(train_json, "r") as f:
    for line in f:
        item = json.loads(line)
        all_train_labels.append(class_to_idx[item["class_label"]])

label_counts  = Counter(all_train_labels)
total_samples = len(all_train_labels)

class_weights = torch.zeros(len(classes))
for idx in range(len(classes)):
    class_weights[idx] = total_samples / (len(classes) * label_counts[idx])

# Added dhoti_pants, increased BOOST_FACTOR to 2.0
CONFUSED_CLASSES = [
    "gowns",
    "women_kurta",
    "mojaris_men",
    "mojaris_women",
    "dhoti_pants",          # newly added
    "leggings_and_salwars",
    "dupattas",
]

BOOST_FACTOR = 2.0          # increased from 1.5

for cls_name in CONFUSED_CLASSES:
    if cls_name in class_to_idx:
        idx = class_to_idx[cls_name]
        class_weights[idx] *= BOOST_FACTOR
        print(f"Boosted weight for '{cls_name}': {class_weights[idx]:.4f}")

class_weights_device = class_weights.to(device)
print("\nClass weights:", class_weights)



In [ ]:
# =========================
# 10. Weighted Random Sampler
#     Over-samples confused classes
#     during training
# =========================

sample_weights = [class_weights[label].item() for label in all_train_labels]
sample_weights = torch.tensor(sample_weights)

sampler = WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = len(sample_weights),
    replacement = True
)



In [ ]:
# =========================
# 11. Datasets & DataLoaders
# =========================

train_dataset = IndoFashionDataset(
    json_path=f"{DATA_DIR}/train_data.json",
    base_path=DATA_DIR,
    transform=train_transform
)

val_dataset = IndoFashionDataset(
    json_path=f"{DATA_DIR}/val_data.json",
    base_path=DATA_DIR,
    transform=test_transform
)

test_dataset = IndoFashionDataset(
    json_path=f"{DATA_DIR}/test_data.json",
    base_path=DATA_DIR,
    transform=test_transform
)

# Use sampler instead of shuffle=True
train_loader = DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    sampler     = sampler,          # Weighted sampling
    num_workers = NUM_WORKERS,
    pin_memory  = True
)

val_loader = DataLoader(
    val_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = True
)

test_loader = DataLoader(
    test_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = True
)

print(f"\nTrain: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
# =========================
# 12. Model
#     Now unfreeze layer2, 3, 4
# =========================

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Freeze all first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze layer2, layer3, layer4, fc
for name, param in model.named_parameters():
    if any(layer in name for layer in ["layer2", "layer3", "layer4", "fc"]):
        param.requires_grad = True

# Better classifier head
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_features, 1024),
    nn.BatchNorm1d(1024),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(1024, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(512, len(classes))
)

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\nTrainable: {trainable:,} / {total:,} params")

In [ ]:
# =========================
# 13. Loss, Optimizer, Scheduler
# =========================

# Focal Loss with class weights — targets hard examples
criterion = FocalLoss(
    gamma           = 2.0,
    weight          = class_weights,
    label_smoothing = 0.1
)

# Differential LRs — deeper layers get smaller LR
optimizer = optim.AdamW([
    {"params": model.layer2.parameters(), "lr": LR_LAYER2},
    {"params": model.layer3.parameters(), "lr": LR_LAYER3},
    {"params": model.layer4.parameters(), "lr": LR_LAYER4},
    {"params": model.fc.parameters(),     "lr": LR_FC},
], weight_decay=1e-4)

# Warmup for first 3 epochs, then cosine decay
def lr_lambda(epoch):
    warmup_epochs = 3
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs      # Linear warmup
    return 1.0

warmup_scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS - 3, eta_min=1e-7
)

scaler = torch.amp.GradScaler('cuda')

In [ ]:
# =========================
# 14. Training Function
#     No MixUp — real train acc
# =========================

def train_model(model, train_loader, val_loader, epochs=EPOCHS):

    train_losses, val_losses         = [], []
    train_accuracies, val_accuracies = [], []

    best_val_acc      = 0.0
    epochs_no_improve = 0

    for epoch in range(epochs):

        # ========================
        # TRAINING
        # ========================
        model.train()
        running_loss = 0.0
        correct = total = 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss    = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            _, predicted  = torch.max(outputs, 1)
            total        += labels.size(0)
            correct      += (predicted == labels).sum().item()

        train_acc  = 100 * correct / total
        train_loss = running_loss / len(train_loader)

        # ========================
        # VALIDATION
        # ========================
        model.eval()
        val_running_loss = 0.0
        val_correct = val_total = 0

        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                images = images.to(device)
                labels = labels.to(device)

                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss    = criterion(outputs, labels)

                val_running_loss += loss.item()
                _, predicted      = torch.max(outputs, 1)
                val_total        += labels.size(0)
                val_correct      += (predicted == labels).sum().item()

        val_acc  = 100 * val_correct / val_total
        val_loss = val_running_loss / len(val_loader)

        # Step scheduler
        if epoch < 3:
            warmup_scheduler.step()
        else:
            cosine_scheduler.step()

        # Store history
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_acc)
        val_accuracies.append(val_acc)

        # Print
        current_lrs = [f"{pg['lr']:.2e}" for pg in optimizer.param_groups]
        print(f"\nEpoch [{epoch+1}/{epochs}]")
        print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.2f}%")
        print(f"  LRs: {current_lrs}")

        # Save best
        if val_acc > best_val_acc:
            best_val_acc      = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), "best_resnet50_fixed.pth")
            print(f"  ✅ Best saved! Val Acc: {best_val_acc:.2f}%")
        else:
            epochs_no_improve += 1
            print(f"  ⚠️  No improvement {epochs_no_improve}/{PATIENCE}")

        # Checkpoint
        torch.save({
            "epoch":                epoch + 1,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_losses":         train_losses,
            "val_losses":           val_losses,
            "train_accuracies":     train_accuracies,
            "val_accuracies":       val_accuracies,
        }, f"checkpoint_epoch_{epoch+1}.pth")

        # Early stopping
        if epochs_no_improve >= PATIENCE:
            print(f"\n🛑 Early stopping at epoch {epoch+1}")
            break

    return train_losses, val_losses, train_accuracies, val_accuracies

In [ ]:
import json

DATA_DIR = "/content/content/balanced_dataset"

labels_set = set()

with open(f"{DATA_DIR}/train_data.json", "r") as f:
    for line in f:
        item = json.loads(line)
        labels_set.add(item["class_label"])

classes      = sorted(list(labels_set))
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

print("Total Classes:", len(classes))
print("\nExact order (index → class):")
for idx, cls in enumerate(classes):
    print(f"  {idx:2d} → {cls}")

In [ ]:
# =========================
# 15. Run Training
# =========================

train_losses, val_losses, train_accuracies, val_accuracies = train_model(
    model, train_loader, val_loader, epochs=EPOCHS
)

In [ ]:
# =========================
# 16. Plot
# =========================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses,      label="Train Loss", marker="o")
axes[0].plot(val_losses,        label="Val Loss",   marker="o")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend(); axes[0].grid(True)

axes[1].plot(train_accuracies,  label="Train Acc",  marker="o")
axes[1].plot(val_accuracies,    label="Val Acc",    marker="o")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig("training_graphs_fixed.png", dpi=150)
plt.show()

In [ ]:
# =========================
# 17. TTA Evaluation
# =========================

def evaluate_with_tta(model, test_dataset, tta_transforms, device):
    """Run TTA — average predictions across multiple transforms."""
    model.eval()

    all_labels      = []
    all_predictions = []
    all_probs       = []

    print("\nRunning TTA evaluation...")

    for idx in tqdm(range(len(test_dataset))):

        item       = test_dataset.data[idx]
        image_path = os.path.join(test_dataset.base_path, item["image_path"])
        raw_image  = Image.open(image_path).convert("RGB")
        true_label = class_to_idx[item["class_label"]]

        tta_probs = []
        with torch.no_grad():
            for tta_tf in tta_transforms:
                img_tensor = tta_tf(raw_image).unsqueeze(0).to(device)
                with torch.amp.autocast('cuda'):
                    output = model(img_tensor)
                prob = F.softmax(output, dim=1).cpu().numpy()[0]
                tta_probs.append(prob)

        avg_prob   = np.mean(tta_probs, axis=0)
        prediction = np.argmax(avg_prob)

        all_labels.append(true_label)
        all_predictions.append(prediction)
        all_probs.append(avg_prob)

    return all_labels, all_predictions, all_probs

In [ ]:
# =========================
# 18. Load Best & Evaluate
# =========================

model.load_state_dict(torch.load("best_resnet50_fixed.pth"))

# --- Standard Evaluation ---
model.eval()
correct = total = 0
std_labels = []
std_preds  = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Standard Eval"):
        images = images.to(device)
        labels = labels.to(device)

        with torch.amp.autocast('cuda'):
            outputs = model(images)

        _, predicted = torch.max(outputs, 1)
        total       += labels.size(0)
        correct     += (predicted == labels).sum().item()
        std_labels.extend(labels.cpu().numpy())
        std_preds.extend(predicted.cpu().numpy())

std_acc = 100 * correct / total
print(f"\n📊 Standard Test Accuracy : {std_acc:.2f}%")

# --- TTA Evaluation ---
tta_labels, tta_preds, tta_probs = evaluate_with_tta(
    model, test_dataset, tta_transforms, device
)
tta_correct = sum(l == p for l, p in zip(tta_labels, tta_preds))
tta_acc     = 100 * tta_correct / len(tta_labels)
print(f"🚀 TTA Test Accuracy      : {tta_acc:.2f}%")
print(f"📈 TTA Gain               : +{tta_acc - std_acc:.2f}%")

In [ ]:
# =========================
# 19. Confusion Matrix
# =========================

cm = confusion_matrix(tta_labels, tta_preds)

plt.figure(figsize=(16, 14))
sns.heatmap(
    cm,
    annot    = True,
    fmt      = "d",
    xticklabels = classes,
    yticklabels = classes,
    cmap     = "Blues"
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title(f"Confusion Matrix (TTA) — Test Acc: {tta_acc:.2f}%")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=45)
plt.tight_layout()
plt.savefig("confusion_matrix_fixed.png", dpi=150)
plt.show()

In [ ]:
# =========================
# 20. Classification Report
# =========================

report = classification_report(tta_labels, tta_preds, target_names=classes)
print("\n" + report)

In [ ]:
# =========================
# 21. Save to Drive
# =========================

import json as json_lib

with open("class_to_idx.json", "w") as f:
    json_lib.dump(class_to_idx, f)

with open("idx_to_class.json", "w") as f:
    json_lib.dump({str(k): v for k, v in idx_to_class.items()}, f)

!mkdir -p /content/drive/MyDrive/resnet50_fixed
!cp best_resnet50_fixed.pth     /content/drive/MyDrive/resnet50_fixed/
!cp *.png                       /content/drive/MyDrive/resnet50_fixed/
!cp class_to_idx.json           /content/drive/MyDrive/resnet50_fixed/
!cp idx_to_class.json           /content/drive/MyDrive/resnet50_fixed/

print("\n✅ Done!")
print(f"   Standard Accuracy : {std_acc:.2f}%")
print(f"   TTA Accuracy      : {tta_acc:.2f}%")
print(f"   Best Val Accuracy : {max(val_accuracies):.2f}%")